# PerCE Demo — 12-lead ECG Counterfactual Explanations

This notebook reproduces the core results from:

> **Bayrak, B. & Bach, K. (2025).** PerCE: Hierarchical Perturbation-Based Counterfactual Explanations for Multivariate Time Series Classification. *IEEE Access*. [DOI: 10.1109/ACCESS.2025.3639125](https://doi.org/10.1109/ACCESS.2025.3639125)

**Dataset:** [CODE-test](https://zenodo.org/records/3765780) — 827 annotated 12-lead ECGs (open access, cardiologist-annotated).

**Task:** Binary classification of 1st-degree AV block (1dAVb).

---
**What you'll see:**
- Load the open ECG dataset from Zenodo
- Train a ResNet classifier
- Generate counterfactual explanations with PerCE
- Reproduce Table 1 from the paper
- Visualise original vs. counterfactual ECG signals

## 0. Install & Import

In [ ]:
# Install dependencies (first run only)
# !pip install perce h5py wfdb matplotlib numpy scipy scikit-learn torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
import urllib.request
import zipfile
import h5py

from perce import PerCEExplainer, evaluate_batch

SEED = 42
np.random.seed(SEED)
print("PerCE demo ready ✓")

## 1. Download CODE-test Dataset

The dataset is hosted on Zenodo and is free to download. It contains 827 12-lead ECG recordings.

In [ ]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ECG_FILE  = DATA_DIR / "ecg_tracings.hdf5"
ANNO_FILE = DATA_DIR / "attributes.csv"

ZENODO_BASE = "https://zenodo.org/records/3765780/files"

def download_if_missing(url, dest):
    if not dest.exists():
        print(f"Downloading {dest.name}...")
        urllib.request.urlretrieve(url, dest)
        print(f"  Saved to {dest}")
    else:
        print(f"  {dest.name} already present.")

download_if_missing(f"{ZENODO_BASE}/ecg_tracings.hdf5", ECG_FILE)
download_if_missing(f"{ZENODO_BASE}/attributes.csv",    ANNO_FILE)

## 2. Load and Prepare Data

In [ ]:
import pandas as pd

# Load ECG signals — shape (N, T, C)
with h5py.File(ECG_FILE, "r") as f:
    X_raw = f["tracings"][:].astype(np.float32)   # (N, T, C)

# Load annotations
anno = pd.read_csv(ANNO_FILE)

print(f"ECG data shape (N, T, C): {X_raw.shape}")
print(f"Annotations shape:        {anno.shape}")
print(f"Columns: {list(anno.columns)}")
print(anno.head())

In [ ]:
# Transpose to (N, C, T) — PerCE convention
X = X_raw.transpose(0, 2, 1)   # (N, C, T)

# Pad to uniform length 4096 (as in the paper)
T_target = 4096
N, C, T_orig = X.shape
if T_orig < T_target:
    pad = T_target - T_orig
    X = np.pad(X, ((0,0), (0,0), (pad//2, pad - pad//2)))
elif T_orig > T_target:
    X = X[:, :, :T_target]

print(f"Final X shape: {X.shape}")

# Labels: 1dAVb (1st degree AV block)
y = anno["1dAVb"].values.astype(int)
print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 3. Train the ResNet Classifier

We use the same ResNet architecture from the paper (Ribeiro et al., 2020).

In [ ]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=7, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel, stride=stride, padding=kernel//2, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel, padding=kernel//2, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        self.skip  = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
            nn.BatchNorm1d(out_ch)
        ) if in_ch != out_ch or stride != 1 else nn.Identity()
        self.relu  = nn.ReLU(inplace=True)
        self.drop  = nn.Dropout(0.2)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.skip(x))


class ECGResNet(nn.Module):
    """Lightweight ResNet for 12-lead ECG binary classification."""
    def __init__(self, n_channels=12):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(n_channels, 64, 7, padding=3, bias=False),
            nn.BatchNorm1d(64), nn.ReLU(inplace=True)
        )
        self.blocks = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 512, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc   = nn.Linear(512, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.pool(x).squeeze(-1)
        return torch.sigmoid(self.fc(x)).squeeze(-1)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_net = ECGResNet(n_channels=C).to(device)
print(f"Model on: {device}")
print(f"Parameters: {sum(p.numel() for p in model_net.parameters()):,}")

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS = 20
BATCH  = 32
LR     = 1e-3

X_tr_t = torch.tensor(X_train, dtype=torch.float32)
y_tr_t = torch.tensor(y_train, dtype=torch.float32)
X_te_t = torch.tensor(X_test,  dtype=torch.float32)
y_te_t = torch.tensor(y_test,  dtype=torch.float32)

loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=BATCH, shuffle=True)

optimizer = torch.optim.Adam(model_net.parameters(), lr=LR)
criterion = nn.BCELoss()

model_net.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_net(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{EPOCHS}  loss={total_loss/len(loader):.4f}")

# Test accuracy
model_net.eval()
with torch.no_grad():
    preds = (model_net(X_te_t.to(device)) > 0.5).cpu().numpy().astype(int)
acc = (preds == y_test).mean()
print(f"\nTest accuracy: {acc:.2%}")

In [ ]:
# Wrap model as a simple callable (N, C, T) → (N,)
def ecg_model(X_np: np.ndarray) -> np.ndarray:
    model_net.eval()
    with torch.no_grad():
        t = torch.tensor(X_np, dtype=torch.float32).to(device)
        probs = model_net(t).cpu().numpy()  # (N,) probabilities
    # Return (N, 2) softmax-like output for compatibility
    return np.stack([1 - probs, probs], axis=1)

# Quick sanity check
sample_pred = ecg_model(X_test[:3])
print("Sample predictions (class probs):", sample_pred)

## 4. Generate Counterfactual Explanations with PerCE

In [ ]:
# Instantiate PerCE with paper parameters
explainer = PerCEExplainer(
    model=ecg_model,
    n_segments=10,
    alpha=0.5,
    beta=0.6,
    k=5,
    dtw_window=0.1,
    random_state=SEED,
)
explainer.fit(X_train, y_train)
print(explainer)

In [ ]:
# Run on 100 test instances (matching the paper evaluation)
N_EVAL = min(100, len(X_test))

# For each test instance: target = opposite class
target_classes = [1 - int(y) for y in y_test[:N_EVAL]]

print(f"Generating {N_EVAL} counterfactuals...")
results = explainer.explain_batch(
    X_test[:N_EVAL],
    target_classes=target_classes,
    verbose=True,
)
print("Done!")

## 5. Reproduce Table 1 from the Paper

In [ ]:
summary = evaluate_batch(results)

print("\n" + "="*55)
print("  PerCE Evaluation Results (Table 1 from paper)")
print("="*55)
print(f"  Instances evaluated : {summary['n_instances']}")
print(f"  Validity rate       : {summary['validity_rate']:.2%}  (paper: 98%)")
print(f"  Proximity (mean±std): {summary['proximity_mean']:.1f} ± {summary['proximity_std']:.1f}  (paper: 50±25)")
print(f"  Sparsity  (mean±std): {summary['sparsity_mean']:.2f} ± {summary['sparsity_std']:.2f}  (paper: 0.40±0.12)")
print(f"  Diversity           : {summary['diversity']:.2f}  (paper: 0.20±0.08)")
print("="*55)

## 6. Visualise: Original vs. Counterfactual ECG

In [ ]:
LEAD_NAMES = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

def plot_ecg_comparison(result, idx=0, n_leads=6, t_start=0, t_end=800):
    """Plot original and counterfactual ECG side by side."""
    orig = result.original
    cf   = result.counterfactual
    t    = np.arange(t_start, t_end)

    fig, axes = plt.subplots(n_leads, 2, figsize=(16, n_leads * 1.8), sharex=True)
    fig.suptitle(
        f"Instance {idx} — Original (class {1 - result.target_class}) "
        f"→ Counterfactual (class {result.target_class})",
        fontsize=13, fontweight="bold"
    )

    for i in range(n_leads):
        ax_orig = axes[i, 0]
        ax_cf   = axes[i, 1]

        ax_orig.plot(t, orig[i, t_start:t_end], color="steelblue", lw=1.0)
        ax_cf.plot(  t, cf[i,   t_start:t_end], color="tomato",    lw=1.0)

        # Highlight modified segments
        if i in result.channels_modified:
            ax_orig.set_facecolor("#fff3f3")
            ax_cf.set_facecolor(  "#fff3f3")

        ax_orig.set_ylabel(LEAD_NAMES[i], fontsize=9)
        ax_orig.tick_params(left=False, labelleft=False)
        ax_cf.tick_params(left=False, labelleft=False)

    axes[0, 0].set_title("Original",        fontsize=11)
    axes[0, 1].set_title("Counterfactual",  fontsize=11)
    axes[-1, 0].set_xlabel("Time (samples)", fontsize=9)
    axes[-1, 1].set_xlabel("Time (samples)", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"ecg_comparison_instance_{idx}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(result.summary())


# Pick sample 2 (highlighted in paper's qualitative analysis)
plot_ecg_comparison(results[2], idx=2)

In [ ]:
# Pick sample 33 (interesting diversity case from the paper)
plot_ecg_comparison(results[33], idx=33)

## 7. Feature Importance Visualisation

In [ ]:
# Show channel and segment importance for one instance
r = results[2]
ch_imp  = r.metadata["channel_importance"]
seg_imp = r.metadata["segment_importance"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Channel importance
bars = ax1.bar(LEAD_NAMES, ch_imp, color=["tomato" if v > 0 else "steelblue" for v in ch_imp])
ax1.axhline(0, color="black", lw=0.8)
ax1.set_title("Channel Feature Importance", fontweight="bold")
ax1.set_xlabel("ECG Lead")
ax1.set_ylabel("Importance (accuracy drop)")

# Segment importance
ax2.plot(range(len(seg_imp)), seg_imp, "o-", color="steelblue", lw=2, ms=6)
ax2.fill_between(range(len(seg_imp)), 0, seg_imp, alpha=0.2, color="steelblue")
ax2.set_title("Segment Feature Importance", fontweight="bold")
ax2.set_xlabel("Segment index")
ax2.set_ylabel("Importance (accuracy drop)")
ax2.set_xticks(range(len(seg_imp)))

plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Distribution of Metrics (Figure 4 from paper)

In [ ]:
proxs  = [r.proximity_score for r in results]
spars  = [r.sparsity_score  for r in results]
valids = [float(r.is_valid) for r in results]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(proxs,  bins=20, color="steelblue", edgecolor="white")
axes[0].axvline(np.median(proxs), color="tomato", lw=2, linestyle="--", label=f"median={np.median(proxs):.1f}")
axes[0].set_title("Proximity Distribution", fontweight="bold")
axes[0].set_xlabel("Proximity score (lower = better)"); axes[0].legend()

axes[1].hist(spars, bins=20, color="steelblue", edgecolor="white")
axes[1].axvline(np.median(spars), color="tomato", lw=2, linestyle="--", label=f"median={np.median(spars):.2f}")
axes[1].set_title("Sparsity Distribution", fontweight="bold")
axes[1].set_xlabel("Sparsity score (higher = fewer changes)"); axes[1].legend()

axes[2].bar(["Invalid", "Valid"], [valids.count(0.0), valids.count(1.0)],
            color=["tomato", "mediumseagreen"], edgecolor="white", width=0.5)
axes[2].set_title("Validity", fontweight="bold")
axes[2].set_ylabel("Count")
axes[2].text(1, valids.count(1.0) + 0.5, f"{np.mean(valids):.1%}", ha="center", fontweight="bold")

plt.suptitle("PerCE Evaluation on CODE-test Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("metric_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Summary

PerCE successfully generates clinically plausible counterfactual explanations for 12-lead ECG classification by:

1. **Identifying which ECG leads and time segments** are most important for the model's decision.
2. **Anchoring to real ECG patterns** from the target class (InSample strategy).
3. **Applying minimal targeted changes** via hierarchical interpolation.

For more, see:
- [`02_custom_model.ipynb`](02_custom_model.ipynb) — using PerCE with your own model
- [`03_CEval_evaluation.ipynb`](03_CEval_evaluation.ipynb) — full evaluation with the CEval toolkit
- [Paper (IEEE Access)](https://doi.org/10.1109/ACCESS.2025.3639125)